# TCRdist neighbor search benchmark

Benchmark sparse neighbor finding under a more advanced sequence similarity metric,
using SymScan as a pre-filter ahead of the exact TCRdist computation. Expected
runtime ~20 min, dominated by the exhaustive reference algorithm.

Writes `../data/tcrdist_benchmark.csv`; plotted by `pub/tcrdist.ipynb`.

In [1]:
import time

import numpy as np
import pandas as pd
import pyrepseq as prs
from tcrdist.repertoire import TCRrep
from tcrdist.rep_funcs import compute_pw_sparse_out_of_memory

import benchutils as bu

# TCRdist thresholds at which recovered pairs are tallied
max_tcrdists = np.arange(0, 52, 3)
# independent subsamples drawn from the repertoire, and their size
nsamples = 1
n_sequence = 1_000
# SymScan pre-filter Levenshtein thresholds
max_editss = [1, 2, 3]

In [2]:
bu.describe_env()

{'colab': False,
 'platform': 'Linux-6.8.0-136-generic-x86_64-with-glibc2.39',
 'python': '3.12.13',
 'git_sha': 'caa6780',
 'cpu_model': '12th Gen Intel(R) Core(TM) i7-1260P',
 'n_cpus_total': 16,
 'affinity': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
 'n_cpus_visible': 16,
 'governor': 'powersave',
 'no_turbo': '0',
 'gpu': {'name': 'NVIDIA T550 Laptop GPU',
  'memory_total': '4096 MiB',
  'clocks_max_sm': '2100 MHz',
  'clocks_applications_gr': '1065 MHz'},
 'thread_env': {'RAYON_NUM_THREADS': None,
  'OMP_NUM_THREADS': None,
  'MKL_NUM_THREADS': None,
  'OPENBLAS_NUM_THREADS': None,
  'NUMEXPR_NUM_THREADS': None,
  'NUMBA_NUM_THREADS': None,
  'OMP_PROC_BIND': None,
  'OMP_PLACES': None},
 'packages': {'symscan': '0.8.3',
  'pyrepseq': '1.6',
  'pybktree': '1.1',
  'rapidfuzz': '3.14.5',
  'pwseqdist': '0.6',
  'numba': '0.66.0',
  'numpy': '2.5.1',
  'scipy': '1.18.0',
  'pandas': '3.0.5'},
 'timeout_seconds': 100}

In [3]:
df = pd.read_csv('../data/emerson_HIP00110.tsv.gz', sep='\t')
df = df[df['amino_acid'].apply(prs.isvalidcdr3)]
df = prs.standardize_dataframe(df, col_mapper={'amino_acid' : 'CDR3B',
                                          'v_family' : 'TRBV',
                                         },
                              suppress_warnings=True)
df = df[df['CDR3B'].apply(prs.isvalidcdr3)]
df.dropna(subset='TRBV', inplace=True)
df.drop_duplicates('CDR3B', inplace=True)
df['CDR3Blen'] = df['CDR3B'].apply(len)
df = df[df['CDR3Blen']>5]
df['TRBV'] = df['TRBV'] + '*01'
df.reset_index(drop=True, inplace=True)

In [4]:
dfs = [df.sample(n_sequence) for i in range(nsamples)]

In [5]:
# warm-up numba for benchmarking
d = dfs[0]
prs.nearest_neighbor_tcrdist(d, max_edits=2,
                             max_tcrdist=0);

In [6]:
rows = []

for max_edits in max_editss:
    for sample, d in enumerate(dfs):
        told = time.time()
        prs_nn = prs.nearest_neighbor_tcrdist(d, max_edits=max_edits,
                                        max_tcrdist=max_tcrdists[-1])
        # tallied inside the timed region, as in the original benchmark
        counts = [int((prs_nn[:, 2]<=dist).sum()) for dist in max_tcrdists]
        runtime_s = time.time()-told
        rows += [{'algorithm': 'symscan', 'max_edits': max_edits, 'sample': sample,
                  'max_tcrdist': int(dist), 'n_neighbors': n, 'runtime_s': runtime_s}
                 for dist, n in zip(max_tcrdists, counts)]

In [7]:
def convert_df_to_tcrdist_form(df: pd.DataFrame):
    mapper = {
            "TRBV": "v_b_gene",
            "CDR3B": "cdr3_b_aa",
            "rearrangement" : 'cdr3_b_nucseq'}
    df = df.rename(columns=mapper)

    df = df[list(mapper.values())]

    if not "count" in df:
        df["count"] = 1

    return df

In [8]:
for sample, d in enumerate(dfs):
    d_tcrdist = convert_df_to_tcrdist_form(d)
    d_tcrdist.reset_index(drop=True, inplace=True)
    tr = TCRrep(cell_df=d_tcrdist, organism='human', chains=['beta'], compute_distances=False)
    told = time.time()
    nn = compute_pw_sparse_out_of_memory(tr, max_distance=max_tcrdists[-1],
                                         pm_pbar=False, row_size=1000, pm_processes=1)[0]
    counts = [int((nn.data<dist+1).sum()-len(d)) for dist in max_tcrdists]
    runtime_s = time.time()-told
    # max_edits does not apply to the exhaustive reference, left empty
    rows += [{'algorithm': 'exhaustive', 'max_edits': pd.NA, 'sample': sample,
              'max_tcrdist': int(dist), 'n_neighbors': n, 'runtime_s': runtime_s}
             for dist, n in zip(max_tcrdists, counts)]

/home/andreas/miniforge3/envs/symdel/lib/python3.12/site-packages/tcrdist/repertoire.py:165: UserWarning: db_file must be 'alphabeta_gammadelta_db.tsv' or 'alphabeta_db.tsv' or 'gammadelta_db.tsv' unless you have built tcrdist3 from scratch
  self._validate_db_file()


CREATED /d81cab32fe3c/ FOR HOLDING DISTANCE OUT OF MEMORY
RETURNING scipy.sparse csr_matrix w/dims (1000, 1000)
CLEANING UP d81cab32fe3c


In [9]:
tcrdist_df = pd.DataFrame(rows)
tcrdist_df.to_csv('../data/tcrdist_benchmark.csv')
tcrdist_df.head()

,algorithm,max_edits,sample,max_tcrdist,n_neighbors,runtime_s
0,symscan,1,0,0,0,0.10667
1,symscan,1,0,3,0,0.10667
2,symscan,1,0,6,0,0.10667
3,symscan,1,0,9,2,0.10667
4,symscan,1,0,12,12,0.10667
